# Build a peptide PSSM with IEGR

Companion to https://tcrchallenge.org/tutorials/iegr-pssm/. Read that guide for input definitions and interpretation. Run the cells in order in the same kernel.

Complete the setup guide first. Run from the DecoderTCR checkout pinned to `3e3d9889d26d79635f674940ddab039c4dc6f9f9`, with the 300M checkpoint downloaded. No model inference has been run in this notebook.

Download https://tcrchallenge.org/downloads/tutorial_helpers.py and place it beside this notebook. Read its documented pairing or PSSM conventions.

For repertoire analysis supply your own `filtered_contig_annotations.csv` and `peptide_targets.csv`. For design workflows inspect the public example and replace it with your chosen context. Record the model, source revision, device, files and settings. Generated sequences are candidates, not validated binders. Source and adapted code: MIT license.


In [ ]:
import DecoderTCR as dt
import pandas as pd
from tutorial_helpers import candidate_pssm

clone = {
    "trav": "TRAV21", "traj": "TRAJ6", "cdr3a": "CAVRPGGAGPFFVVF",
    "trbv": "TRBV7-9", "trbj": "TRBJ2-7", "cdr3b": "CASSLGQAYEQYF",
    "hla": "HLA-B*27:05",
}
# Teaching settings: inspect and change for your analysis.
designs = dt.design_peptides(
    clone, length=9, n=10, method="iegr", temperature=0.1,
    seed=42, gibbs_rounds=10, gibbs_subset_size=5,
    rescore=True, from_genes=True,
    model="DecoderTCR-ESMC_300M", device="cpu",
)
designs.to_csv("iegr_candidates.csv", index=False)
print(f"Returned {len(designs)} unique candidates.")

In [ ]:
counts, frequencies, pssm = candidate_pssm(
    designs["sequence"].tolist(), pseudocount_mass=1.0
)
for name, rows in [("counts", counts), ("frequencies", frequencies), ("pssm", pssm)]:
    pd.DataFrame(rows).to_csv(f"iegr_{name}.csv", index=False)
print(pd.DataFrame(pssm))

In [ ]:
import json
from pathlib import Path
run_record = {
    "source_revision": "3e3d9889d26d79635f674940ddab039c4dc6f9f9",
    "model": "DecoderTCR-ESMC_300M", "device": "cpu",
    "guide": "https://tcrchallenge.org/tutorials/iegr-pssm/", "notebook": "iegr-pssm",
    "note": "Save this notebook with the actual inputs and generation settings used."
}
Path("iegr-pssm-run.json").write_text(json.dumps(run_record, indent=2))